# Data cleaning

This notebook owns every cleaning rule. Original fields are retained alongside conservative Unicode text for Qwen and separate transliterated retrieval fields. Numeric strings retain leading zeros. Legal suffix removal is an alternate name view; short names such as MS, PC and LP remain intact. Processing is batched into versioned Parquet with explicit schemas, checksums and a manifest.

In [ ]:
from pathlib import Path
import json
_ROOT = Path(__file__).resolve().parent if "__file__" in globals() else next(
    p / "anshp" for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "anshp" / "config.json").is_file())
_A = {"__file__": str(_ROOT / "00_setup_and_data_audit.ipynb")}
for _c in json.loads((_ROOT / "00_setup_and_data_audit.ipynb").read_text(encoding="utf-8"))["cells"]:
    if _c["cell_type"] == "code" and "er-definitions" in _c.get("metadata", {}).get("tags", []):
        exec("".join(_c["source"]), _A)

import re
import unicodedata

# Suffix stripping is a retrieval alternative only. Never replace the preserved name.
LEGAL_SUFFIXES = {"inc", "incorporated", "llc", "ltd", "limited", "corp", "corporation",
                  "llp", "plc", "pvt", "private", "sarl", "sas", "gmbh"}
NORMALIZATION_VERSION = "normalization-v2"
_ASCII_NEEDS_FTFY = re.compile(r"[&\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")

def conservative_text(value):
    import ftfy
    if value is None:
        return ""
    if not isinstance(value, str):
        raise TypeError("Text columns must be strings or None")
    if value.isascii() and not _ASCII_NEEDS_FTFY.search(value):
        return " ".join(value.split())
    text = unicodedata.normalize("NFKC", ftfy.fix_text(value))
    return " ".join("".join(ch if not unicodedata.category(ch).startswith("C")
                            else " " for ch in text).split())

def lexical_text(value):
    from anyascii import anyascii
    return " ".join(re.findall(r"[a-z0-9]+", (value if value.isascii() else anyascii(value)).casefold()))

def normalize_record(record):
    name = conservative_text(record["business_name"])
    address = conservative_text(record["business_address"])
    lex_name, lex_address = lexical_text(name), lexical_text(address)
    words = lex_name.split()
    core = list(words)
    while len(core) > 1 and core[-1] in LEGAL_SUFFIXES:
        core.pop()
    joined = "".join(words)
    # Include whole joined short names so "MS" and "M S" can meet in this channel.
    initials = "".join(w[0] for w in words if w)
    variants = sorted({"i" + initials, "i" + joined} - {"i"}) if len(joined) <= 8 \
               else (["i" + initials] if initials else [])
    grams = sorted({joined[i:i+3] for i in range(max(0, len(joined)-2))})
    return {"entity_id": record["entity_id"], "country": record["country"],
            "raw_name": record["business_name"], "raw_address": record["business_address"],
            "business_name": name, "business_address": address,
            "lex_name": lex_name, "name_core": " ".join(core),
            "lex_address": lex_address, "grams": " ".join("g"+g for g in grams),
            "initials": " ".join(variants),
            "numbers": " ".join(sorted({"n"+n for n in re.findall(r"\d+", lex_address)}))}

def normalized_schema():
    import pyarrow as pa
    return pa.schema([(k, pa.string()) for k in
        ("entity_id", "country", "raw_name", "raw_address", "business_name",
         "business_address", "lex_name", "name_core", "lex_address",
         "grams", "initials", "numbers")])

def build_normalized(audit_dir, output_dir, batch_size=20000, progress=None):
    import pyarrow as pa
    import pyarrow.parquet as pq
    audit_dir, out = Path(audit_dir), Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    audit = _A["require_manifest"](audit_dir / "manifest.json", "audit")
    identity = {"version": NORMALIZATION_VERSION, "audit": audit["fingerprint"],
                "helpers": _A["load_helpers"](_ROOT / "01_data_cleaning.ipynb")[1],
                "packages": {k: _A["importlib"].metadata.version(k) for k in ("ftfy","anyascii")}}
    fingerprint = _A["stable_hash"](identity)
    manifest_path = out / "manifest.json"
    if manifest_path.exists():
        old = _A["require_manifest"](manifest_path, "normalized")
        if old["fingerprint"] == fingerprint:
            return old
    con = _A["connect_database"](audit_dir / audit["database"], read_only=True)
    files = []
    started = _A["time"].perf_counter()
    def emit(event):
        event.update(stage="normalization",fingerprint=fingerprint,seconds=_A["time"].perf_counter()-started)
        _A["atomic_json"](out/"progress.json",event)
        if progress is not None:progress(event)
    try:
        for dataset in ("train", "test"):
            for source in (1,2,3):
                target = out / f"{dataset}-source{source}-{fingerprint[:16]}.parquet"
                checkpoint = target.with_suffix(".complete.json")
                if checkpoint.exists() and target.exists():
                    saved = json.loads(checkpoint.read_text(encoding="utf-8"))
                    item = saved["file"]
                    if saved["fingerprint"] == fingerprint and target.stat().st_size == item["bytes"] and _A["sha256_file"](target) == item["sha256"]:
                        files.append(item)
                        emit({"dataset":dataset,"source":source,"rows":item["rows"],"state":"reused"})
                        continue
                temporary = target.with_suffix(".tmp.parquet")
                cursor = con.execute("""SELECT entity_id,business_name,business_address,country
                    FROM records WHERE dataset=? AND source=?""", [dataset,source])
                count = 0
                with pq.ParquetWriter(temporary, normalized_schema(), compression="zstd") as writer:
                    while rows := cursor.fetchmany(batch_size):
                        records = [normalize_record(dict(zip(
                            ("entity_id","business_name","business_address","country"),r))) for r in rows]
                        writer.write_table(pa.Table.from_pylist(records, schema=normalized_schema()))
                        count += len(records)
                        if count % 100000 == 0:
                            emit({"dataset":dataset,"source":source,"rows":count,"state":"running"})
                temporary.replace(target)
                item = _A["file_entry"](target, out)
                item.update(dataset=dataset, source=source, rows=count)
                files.append(item)
                _A["atomic_json"](checkpoint,{"fingerprint":fingerprint,"file":item})
                emit({"dataset":dataset,"source":source,"rows":count,"state":"file_complete"})
    finally:
        con.close()
    manifest = {"stage": "normalized", "complete": True, "fingerprint": fingerprint,
                "identity": identity, "files": files, "schema": str(normalized_schema())}
    _A["atomic_json"](manifest_path, manifest)
    emit({"state":"complete","rows":sum(f["rows"] for f in files)})
    return manifest

def normalized_paths(normalized_dir, dataset, sources):
    normalized_dir = Path(normalized_dir)
    manifest = _A["require_manifest"](normalized_dir / "manifest.json", "normalized",
                                      verify_hashes=False)
    return [normalized_dir / x["path"] for x in manifest["files"]
            if x["dataset"] == dataset and x["source"] in sources]


In [ ]:
for short in ("MS", "PC", "LP"):
    r = normalize_record(dict(entity_id="x", business_name=short,
                             business_address="0012 Main St", country="US"))
    assert r["name_core"] == short.lower() and r["numbers"] == "n0012"
r = normalize_record(dict(entity_id="x", business_name="École भारत",
                         business_address="12 Rue de l’École", country="FR"))
assert "भारत" in r["business_name"] and "ecole" in r["lex_name"]
assert normalize_record(dict(entity_id="x", business_name="", business_address="", country="US"))["name_core"] == ""
print("Cleaning checks passed")


In [ ]:
from pathlib import Path
import json
PROJECT_DIR = next(p / "anshp" for p in (Path.cwd(), *Path.cwd().parents)
                   if (p / "anshp" / "config.json").is_file())
DATA_DIR = PROJECT_DIR.parent / "6ab10eb3b23ba_student_resource" / "student_resource" / "dataset"
ARTIFACT_DIR = PROJECT_DIR / "artifacts"
CONFIG = json.loads((PROJECT_DIR / "config.json").read_text(encoding="utf-8"))

RUN_CLEANING = False
if RUN_CLEANING:
    normalized = build_normalized(ARTIFACT_DIR / 'audit', ARTIFACT_DIR / 'normalized')
    print([(x['dataset'],x['source'],x['rows']) for x in normalized['files']])
